# Tier 3 Solutions Set 4: New Topics (19–23)

Solutions for Assignment Set 4: New Topics.

In [ ]:
import numpy as np
import math
import random

## Problem 1: GWAS Chi-Square Association Test (★, 10 pts)

In [ ]:
def gwas_chi2_test(contingency_table: list[list[int]]) -> dict:
    """
    Compute GWAS association statistics from a 2x2 allele count table.

    Args:
        contingency_table: [[case_ref, case_alt], [ctrl_ref, ctrl_alt]]

    Returns:
        Dict with keys 'chi2', 'alt_freq_cases', 'alt_freq_controls', 'odds_ratio',
        all rounded to 4 decimal places.
    """
    a = contingency_table[0][0]  # case_ref
    b = contingency_table[0][1]  # case_alt
    c = contingency_table[1][0]  # ctrl_ref
    d = contingency_table[1][1]  # ctrl_alt
    N = a + b + c + d

    chi2 = N * (a*d - b*c)**2 / ((a+b) * (c+d) * (a+c) * (b+d))
    alt_freq_cases    = b / (a + b)
    alt_freq_controls = d / (c + d)
    odds_ratio        = (b * c) / (a * d)

    return {
        "chi2":               round(chi2, 4),
        "alt_freq_cases":     round(alt_freq_cases, 4),
        "alt_freq_controls":  round(alt_freq_controls, 4),
        "odds_ratio":         round(odds_ratio, 4),
    }


# Test
table = [[480, 120], [450, 50]]
result = gwas_chi2_test(table)
print(result)

## Problem 2: Genomic Inflation Factor (★★, 10 pts)

In [ ]:
def genomic_inflation(chi2_values: list[float]) -> dict:
    """
    Compute the genomic inflation factor and apply genomic control correction.

    Args:
        chi2_values: List of chi-square statistics (1 df each) from a GWAS.

    Returns:
        Dict with keys:
            'lambda': genomic inflation factor (4 dp)
            'corrected_chi2': list of chi2 values divided by lambda (each 4 dp)
    """
    # Implement median from scratch
    sorted_vals = sorted(chi2_values)
    n = len(sorted_vals)
    if n % 2 == 1:
        median = sorted_vals[n // 2]
    else:
        median = (sorted_vals[n//2 - 1] + sorted_vals[n//2]) / 2.0

    lam = median / 0.4549
    corrected = [round(v / lam, 4) for v in chi2_values]

    return {
        "lambda":         round(lam, 4),
        "corrected_chi2": corrected,
    }


# Test: inflate chi2 values so lambda > 1
random.seed(42)
chi2_vals = [random.uniform(0.1, 4.0) * 1.3 for _ in range(20)]
result = genomic_inflation(chi2_vals)
print(f"Lambda: {result['lambda']}")
print(f"First 5 corrected chi2: {result['corrected_chi2'][:5]}")

## Problem 3: Spatial Neighbor Graph (★, 10 pts)

In [ ]:
def spatial_neighbor_graph(coords: list[tuple[float, float]], r: float) -> dict:
    """
    Build a spatial neighbor graph based on Euclidean distance.

    Args:
        coords: List of (x, y) coordinate tuples for each spot.
        r: Radius threshold; spots within this distance are neighbors.

    Returns:
        Dict with keys:
            'adjacency': {spot_index: [sorted neighbor indices]}
            'avg_neighbors': float (4 dp)
    """
    n = len(coords)
    adjacency = {}

    for i in range(n):
        neighbors = []
        for j in range(n):
            if i == j:
                continue
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            dist = math.sqrt(dx*dx + dy*dy)
            if dist <= r:
                neighbors.append(j)
        adjacency[i] = sorted(neighbors)

    total_neighbors = sum(len(v) for v in adjacency.values())
    avg_neighbors = total_neighbors / n if n > 0 else 0.0

    return {
        "adjacency":     adjacency,
        "avg_neighbors": round(avg_neighbors, 4),
    }


# Test
coords = [(0, 0), (1, 0), (0, 1), (5, 5), (5.5, 5)]
result = spatial_neighbor_graph(coords, r=1.5)
print("Adjacency:", result['adjacency'])
print("Avg neighbors:", result['avg_neighbors'])

## Problem 4: Spatial Autocorrelation — Moran's I (★★, 15 pts)

In [ ]:
def morans_i(expression: list[float], adjacency: dict) -> float:
    """
    Compute Moran's I spatial autocorrelation statistic.

    Args:
        expression: List of expression values, one per spot.
        adjacency: Dict mapping spot index to list of neighbor indices.

    Returns:
        Moran's I statistic (float, 4 dp).
    """
    N = len(expression)
    W = sum(len(v) for v in adjacency.values())

    mean_x = sum(expression) / N
    deviations = [x - mean_x for x in expression]

    numerator = 0.0
    for i in range(N):
        for j in adjacency.get(i, []):
            numerator += deviations[i] * deviations[j]

    denominator = sum(d**2 for d in deviations)

    I = (N / W) * (numerator / denominator)
    return round(I, 4)


# Test: strong spatial clustering — neighbors have similar high/low values
expr = [10.0, 10.5, 9.8, 1.0, 1.2]
adj  = {0: [1, 2], 1: [0, 2], 2: [0, 1], 3: [4], 4: [3]}
i_stat = morans_i(expr, adj)
print(f"Moran's I: {i_stat}  (expect positive, near 1)")

## Problem 5: Log-Ratio Copy Number Segmentation (★★, 15 pts)

In [ ]:
def segment_copy_numbers(
    log2_ratios: list[float],
    window: int = 5,
    threshold: float = 0.3,
) -> list[tuple[int, int, float]]:
    """
    Segment log2 copy-number ratios using a sliding-window CBS-like approach.

    Args:
        log2_ratios: List of log2 copy-number ratios, one per genomic bin.
        window: Sliding window size for smoothing.
        threshold: Minimum absolute difference between consecutive smoothed
                   values to call a breakpoint.

    Returns:
        List of (start_idx, end_idx, segment_mean) tuples.
        Indices are 0-based and inclusive; segment_mean is rounded to 4 dp.
    """
    n = len(log2_ratios)

    # Sliding window means (full windows only)
    # i-th smoothed value = mean of bins i..i+window-1, center = i+window//2
    smoothed = []
    centers  = []
    for i in range(n - window + 1):
        win_mean = sum(log2_ratios[i:i+window]) / window
        smoothed.append(win_mean)
        centers.append(i + window // 2)

    # Find breakpoints: positions in centers array where |smoothed[i+1] - smoothed[i]| > threshold
    breakpoint_centers = []
    for i in range(len(smoothed) - 1):
        if abs(smoothed[i+1] - smoothed[i]) > threshold:
            # Breakpoint falls between centers[i] and centers[i+1]
            breakpoint_centers.append(centers[i] + 1)

    # Build segments from breakpoints
    boundaries = [0] + breakpoint_centers + [n]
    segments = []
    for idx in range(len(boundaries) - 1):
        start = boundaries[idx]
        end   = boundaries[idx+1] - 1
        if start > end:
            continue
        seg_mean = sum(log2_ratios[start:end+1]) / (end - start + 1)
        segments.append((start, end, round(seg_mean, 4)))

    return segments


# Test: flat region, then gain, then loss
ratios = [0.0] * 10 + [1.2] * 10 + [-0.8] * 10
segments = segment_copy_numbers(ratios)
for seg in segments:
    print(f"  [{seg[0]}:{seg[1]}]  mean={seg[2]}")

## Problem 6: Bayes Factor for Variant Pathogenicity (★★, 15 pts)

In [ ]:
def bayesian_pathogenicity(
    prior: float,
    sensitivity: float,
    specificity: float,
    results: list[str],
) -> float:
    """
    Sequentially update the probability of variant pathogenicity.

    Args:
        prior: Prior probability the variant is pathogenic (0–1).
        sensitivity: P(positive test | pathogenic).
        specificity: P(negative test | benign).
        results: Ordered list of test outcomes: 'positive' or 'negative'.

    Returns:
        Final posterior probability of pathogenicity (float, 4 dp).
    """
    P = prior
    sens = sensitivity
    spec = specificity

    for result in results:
        if result == 'positive':
            # P(pos|path)*P(path) / P(pos)
            P = (sens * P) / (sens * P + (1 - spec) * (1 - P))
        else:  # 'negative'
            # P(neg|path)*P(path) / P(neg)
            P = ((1 - sens) * P) / ((1 - sens) * P + spec * (1 - P))

    return round(P, 4)


# Test
posterior = bayesian_pathogenicity(
    prior=0.1,
    sensitivity=0.9,
    specificity=0.95,
    results=['positive', 'positive', 'negative'],
)
print(f"Final posterior: {posterior}")

## Problem 7: ATAC-seq Fragment Size Classification (★, 15 pts)

In [ ]:
def classify_atac_fragments(fragment_lengths: list[int]) -> dict:
    """
    Classify ATAC-seq fragment lengths into nucleosomal categories.

    Args:
        fragment_lengths: List of fragment lengths in base pairs.

    Returns:
        Dict with keys 'sub_nuc', 'mono_nuc', 'di_nuc', 'tri_nuc' (int counts),
        'nfr_ratio' (float, 4 dp), and 'total' (int).
    """
    sub_nuc  = 0
    mono_nuc = 0
    di_nuc   = 0
    tri_nuc  = 0

    for length in fragment_lengths:
        if length < 147:
            sub_nuc  += 1
        elif length <= 294:
            mono_nuc += 1
        elif length <= 441:
            di_nuc   += 1
        else:
            tri_nuc  += 1

    total     = len(fragment_lengths)
    nfr_ratio = sub_nuc / total if total > 0 else 0.0

    return {
        "sub_nuc":  sub_nuc,
        "mono_nuc": mono_nuc,
        "di_nuc":   di_nuc,
        "tri_nuc":  tri_nuc,
        "nfr_ratio": round(nfr_ratio, 4),
        "total":    total,
    }


# Test
random.seed(7)
fragments = (
    [random.randint(50, 146) for _ in range(50)]   # sub-nuc
    + [random.randint(147, 294) for _ in range(30)] # mono-nuc
    + [random.randint(295, 441) for _ in range(15)] # di-nuc
    + [random.randint(442, 600) for _ in range(5)]  # tri-nuc
)
result = classify_atac_fragments(fragments)
print(result)

## Problem 8: TF Footprint Score (★★★, 10 pts)

In [ ]:
def tf_footprint_score(
    insertion_profiles: list[list[float]],
    flank_half: int = 10,
) -> dict:
    """
    Compute a TF footprint score from per-site Tn5 insertion profiles.

    Args:
        insertion_profiles: 2D array of shape (n_sites, n_positions).
                            Each row is the insertion count profile for one site.
        flank_half: Half-width of the central footprint region and each
                    flanking window (default 10).

    Returns:
        Dict with keys:
            'footprint_score': float (4 dp)
            'avg_profile': list of floats (4 dp each)
    """
    # Average across sites (axis=0) to get 1D profile
    n_sites = len(insertion_profiles)
    n_positions = len(insertion_profiles[0])

    profile = [0.0] * n_positions
    for site in insertion_profiles:
        for j in range(n_positions):
            profile[j] += site[j]
    profile = [v / n_sites for v in profile]

    center = n_positions // 2

    # Extract regions
    central     = profile[center - flank_half : center + flank_half]
    left_flank  = profile[center - 3*flank_half : center - flank_half]
    right_flank = profile[center + flank_half : center + 3*flank_half]

    flanks   = left_flank + right_flank
    mean_flanks  = sum(flanks) / len(flanks) if flanks else 0.0
    mean_central = sum(central) / len(central) if central else 0.0

    footprint_score = mean_flanks - mean_central

    return {
        "footprint_score": round(footprint_score, 4),
        "avg_profile":     [round(v, 4) for v in profile],
    }


# Test: simulate a clear footprint (low centre, high flanks)
import numpy as np
np.random.seed(0)
n_sites, n_pos = 50, 100
profiles = np.random.poisson(lam=5, size=(n_sites, n_pos)).astype(float)
# Deplete centre ±10 around position 50
profiles[:, 40:60] *= 0.2
result = tf_footprint_score(profiles.tolist())
print(f"Footprint score: {result['footprint_score']}  (expect positive)")
print(f"Avg profile length: {len(result['avg_profile'])}")